In [1]:
# Cell 1 — Configuration

from pathlib import Path
from PIL import Image, ImageOps
import pandas as pd
import json, re, shutil

EXPORTS_ROOT = Path("/Users/jwatts/Documents/astrophotography/Exports")
DSO_JSON     = Path("mobile/dso.json")  # existing catalog with metadata
OUTPUT_DIR   = Path("web")
JPEG_QUALITY = 95
MAX_FILE_SIZE = 10 * 1024 * 1024  # 10 MB Cloudinary free tier limit
MIN_QUALITY  = 80                  # never go below this
IMAGE_EXTENSIONS = {".jpg", ".jpeg"}

# Set to True when you need to upload images to Cloudinary (e.g. new images added)
UPLOAD_TO_CLOUDINARY = True

# Cloudinary config
CLOUDINARY_ROOT = "island-skies-astro"
CLOUDINARY_FOLDER_MAP = {
    "dso":          "dso",
    "solar-system": "solarsystem",
}

# Map Exports folders → website categories
CATEGORY_MAP = {
    "DSO":    "dso",
    "Planet": "solar-system",
    "Comet":  "solar-system",
    "Lunar":  "solar-system",
}

print(f"Exports root: {EXPORTS_ROOT}")
print(f"Output dir:   {OUTPUT_DIR.resolve()}")
print(f"Cloudinary upload: {'ENABLED' if UPLOAD_TO_CLOUDINARY else 'DISABLED'}")

Exports root: /Users/jwatts/Documents/astrophotography/Exports
Output dir:   /Users/jwatts/StudioProjects/git/AstroPlannerData/web
Cloudinary upload: ENABLED


In [2]:
# Cell 2 — Helper functions

def to_kebab(s: str) -> str:
    """CamelCase/underscores/hyphens → kebab-case."""
    s = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', '-', s)
    s = s.replace('_', '-')
    return s.lower()


def make_image_id(object_id: str, region: str | None, filename_stem: str) -> str:
    """Generate a unique, URL-friendly image ID."""
    stem = to_kebab(filename_stem)
    if region:
        return f"{object_id}-{to_kebab(region)}"
    if stem == object_id or stem.replace('-', '') == object_id:
        return object_id
    if stem.startswith(object_id):
        return stem
    return f"{object_id}-{stem}"


def format_catalog_id(object_id: str) -> str:
    """ic443 → IC 443, m42 → M 42, ngc7000 → NGC 7000, barnard68 → Barnard 68."""
    match = re.match(r'^([a-zA-Z]+)(\d+)$', object_id)
    if match:
        prefix, number = match.groups()
        if prefix.lower() in ('m', 'ngc', 'ic'):
            return f"{prefix.upper()} {number}"
        return f"{prefix.capitalize()} {number}"
    return object_id


# Quick sanity checks
assert to_kebab("PillarsOfCreation") == "pillars-of-creation"
assert to_kebab("IC1805Inner") == "ic1805-inner"
assert to_kebab("jupiter_grs_io") == "jupiter-grs-io"
assert make_image_id("m16", "PillarsOfCreation", "pillars") == "m16-pillars-of-creation"
assert make_image_id("ic434", None, "IC434") == "ic434"
assert make_image_id("ic434", None, "IC4342") == "ic4342"
assert make_image_id("ic410", None, "tadpoles") == "ic410-tadpoles"
assert make_image_id("m42", None, "m42-2") == "m42-2"
assert format_catalog_id("ic443") == "IC 443"
assert format_catalog_id("ngc7000") == "NGC 7000"
assert format_catalog_id("barnard68") == "Barnard 68"
print("All checks passed.")

All checks passed.


In [3]:
# Cell 3 — Discovery functions

def discover_dso() -> list[dict]:
    """Walk DSO/ recursively for all .jpg files.
    Direct images: DSO/<Object>/<file>.jpg
    Region images: DSO/<Object>/<Region>/<file>.jpg
    """
    results = []
    dso_root = EXPORTS_ROOT / "DSO"
    if not dso_root.exists():
        print(f"WARNING: {dso_root} not found")
        return results

    for jpg in sorted(dso_root.rglob("*")):
        if not jpg.is_file() or jpg.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        rel = jpg.relative_to(dso_root)
        parts = rel.parts

        object_id = parts[0].lower()

        if len(parts) == 2:
            region = None
        elif len(parts) == 3:
            region = parts[1]
        else:
            print(f"  WARNING: unexpected depth {rel}")
            continue

        image_id = make_image_id(object_id, region, jpg.stem)

        results.append({
            "imageId": image_id,
            "objectId": object_id,
            "region": region,
            "category": "dso",
            "src_path": str(jpg),
            "filename": jpg.name,
        })

    return results


def discover_planets() -> list[dict]:
    """Walk Planet/ — session subfolders become separate images.
    Direct images: Planet/<Planet>/<file>.jpg → imageId = planet name
    Session folders: Planet/<Planet>/<session>/<file>.jpg → imageId = session name
    """
    results = []
    planet_root = EXPORTS_ROOT / "Planet"
    if not planet_root.exists():
        print(f"WARNING: {planet_root} not found")
        return results

    for planet_folder in sorted(planet_root.iterdir()):
        if not planet_folder.is_dir():
            continue
        planet_name = planet_folder.name.lower()

        # Direct images in planet folder
        direct_images = sorted(
            f for f in planet_folder.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        )
        for i, img in enumerate(direct_images):
            img_id = planet_name if len(direct_images) == 1 else f"{planet_name}-{i+1}"
            results.append({
                "imageId": img_id,
                "objectId": planet_name,
                "region": None,
                "category": "solar-system",
                "src_path": str(img),
                "filename": img.name,
            })

        # Session subfolders
        for session_folder in sorted(planet_folder.iterdir()):
            if not session_folder.is_dir():
                continue
            session_name = to_kebab(session_folder.name)
            img = next(
                (f for f in sorted(session_folder.iterdir())
                 if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS),
                None,
            )
            if img:
                results.append({
                    "imageId": session_name,
                    "objectId": planet_name,
                    "region": None,
                    "category": "solar-system",
                    "src_path": str(img),
                    "filename": img.name,
                })

    return results


def discover_lunar() -> list[dict]:
    """Walk Lunar/<body>/<feature>/ — each feature subfolder is one image."""
    results = []
    lunar_root = EXPORTS_ROOT / "Lunar"
    if not lunar_root.exists():
        print(f"WARNING: {lunar_root} not found")
        return results

    for body_folder in sorted(lunar_root.iterdir()):
        if not body_folder.is_dir():
            continue
        body_name = body_folder.name.lower()

        for feature_folder in sorted(body_folder.iterdir()):
            if not feature_folder.is_dir():
                continue
            feature_name = feature_folder.name.lower()
            img = next(
                (f for f in sorted(feature_folder.iterdir())
                 if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS),
                None,
            )
            if img:
                results.append({
                    "imageId": f"{body_name}-{feature_name}",
                    "objectId": body_name,
                    "region": feature_name,
                    "category": "solar-system",
                    "src_path": str(img),
                    "filename": img.name,
                })

    return results


def discover_comets() -> list[dict]:
    """Walk Comet/<name>/ — one image per comet folder."""
    results = []
    comet_root = EXPORTS_ROOT / "Comet"
    if not comet_root.exists():
        print(f"WARNING: {comet_root} not found")
        return results

    for comet_folder in sorted(comet_root.iterdir()):
        if not comet_folder.is_dir():
            continue
        comet_name = comet_folder.name.lower()
        img = next(
            (f for f in sorted(comet_folder.iterdir())
             if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS),
            None,
        )
        if img:
            results.append({
                "imageId": comet_name,
                "objectId": comet_name,
                "region": None,
                "category": "solar-system",
                "src_path": str(img),
                "filename": img.name,
            })

    return results


print("Discovery functions defined.")

Discovery functions defined.


In [4]:
# Cell 4 — Run discovery, build DataFrame, join with DSO catalog

all_images = []
all_images.extend(discover_dso())
all_images.extend(discover_planets())
all_images.extend(discover_lunar())
all_images.extend(discover_comets())

images_df = pd.DataFrame(all_images)

# Load DSO catalog for metadata
dso_catalog = pd.read_json(DSO_JSON)
dso_meta = dso_catalog[["objectId", "displayName", "ra", "dec", "constellation", "type", "subType"]].copy()

# Left join to get DSO metadata
images_df = images_df.merge(dso_meta, on="objectId", how="left")

# Generate display names
def make_display_name(row):
    base_name = row.get("displayName")
    if pd.notna(base_name):
        if pd.notna(row.get("region")) and row["category"] == "dso":
            # Add human-readable region name for DSO region images
            region_pretty = re.sub(r'(?<=[a-z0-9])(?=[A-Z])', ' ', str(row["region"]))
            return f"{base_name} - {region_pretty}"
        return base_name
    # Fallback: capitalize objectId
    return row["objectId"].replace("-", " ").title()

images_df["displayName"] = images_df.apply(make_display_name, axis=1)

# Print summary
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)
print(f"{'imageId':<35} {'category':<15} {'objectId':<12} {'displayName'}")
print("-" * 110)
for _, row in images_df.iterrows():
    print(f"  {row['imageId']:<33} {row['category']:<15} {row['objectId']:<12} {row['displayName']}")

dso_count = len(images_df[images_df["category"] == "dso"])
ss_count = len(images_df[images_df["category"] == "solar-system"])
print(f"\nTotal: {len(images_df)} images ({dso_count} DSO, {ss_count} solar-system)")

# Check for duplicate imageIds
dupes = images_df[images_df.duplicated(subset="imageId", keep=False)]
if len(dupes) > 0:
    print(f"\nWARNING: {len(dupes)} duplicate imageIds:")
    print(dupes[["imageId", "src_path"]])

imageId                             category        objectId     displayName
--------------------------------------------------------------------------------------------------------------
  ic1805                            dso             ic1805       Heart Nebula
  ic1805-ic1805-inner               dso             ic1805       Heart Nebula - IC1805 Inner
  ic410                             dso             ic410        Tadpoles Nebula
  ic410-tadpoles                    dso             ic410        Tadpoles Nebula
  ic434-2                           dso             ic434        Horsehead Nebula
  ic434-3                           dso             ic434        Horsehead Nebula
  ic434                             dso             ic434        Horsehead Nebula
  ic443                             dso             ic443        Jellyfish Nebula
  ic5146                            dso             ic5146       Cocoon Nebula
  m1                                dso             m1           Crab Ne

In [5]:
# Cell 5 — Process images (DSO: JPEG re-compression; solar-system: copy as-is)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "dso").mkdir(exist_ok=True)
(OUTPUT_DIR / "solar-system").mkdir(exist_ok=True)

widths = []
heights = []

for _, row in images_df.iterrows():
    src = Path(row["src_path"])
    dst = OUTPUT_DIR / row["category"] / f"{row['imageId']}.jpg"

    with Image.open(src) as img:
        img = ImageOps.exif_transpose(img)
        w, h = img.size
        widths.append(w)
        heights.append(h)

        if row["category"] == "solar-system":
            # Planetary/lunar/comet images are small — copy original unchanged
            shutil.copy2(src, dst)
            size_mb = dst.stat().st_size / (1024 * 1024)
            print(f"  {row['imageId']:<35} {w}x{h}  →  {size_mb:.1f} MB (copied)")
        else:
            # DSO images: re-compress, step down quality if over 10 MB
            quality = JPEG_QUALITY
            while quality >= MIN_QUALITY:
                img.save(dst, format="JPEG", quality=quality, optimize=True)
                file_size = dst.stat().st_size
                if file_size <= MAX_FILE_SIZE:
                    break
                quality -= 2

            size_mb = file_size / (1024 * 1024)
            flag = " ⚠️ OVER 10MB" if file_size > MAX_FILE_SIZE else ""
            q_note = f" (q={quality})" if quality != JPEG_QUALITY else ""
            print(f"  {row['imageId']:<35} {w}x{h}  →  {size_mb:.1f} MB{q_note}{flag}")

images_df["width"] = widths
images_df["height"] = heights

print(f"\nDone. {len(images_df)} images written to {OUTPUT_DIR.resolve()}")

  ic1805                              6240x4162  →  8.5 MB
  ic1805-ic1805-inner                 4144x2822  →  1.6 MB
  ic410                               4052x2674  →  1.6 MB
  ic410-tadpoles                      4144x2822  →  1.6 MB
  ic434-2                             2800x2800  →  0.9 MB
  ic434-3                             7356x2780  →  2.9 MB
  ic434                               4100x2775  →  1.7 MB
  ic443                               6238x4167  →  8.2 MB
  ic5146                              4136x2811  →  2.7 MB
  m1                                  3907x2585  →  1.7 MB
  m101                                4129x2811  →  1.6 MB
  m13                                 4138x2820  →  6.3 MB
  m16-pillars-of-creation             4144x2822  →  1.1 MB
  m16                                 4134x2815  →  2.2 MB
  m20                                 4118x2791  →  2.4 MB
  m27                                 4126x2814  →  3.1 MB
  m3                                  4134x2818  →  2.1 

In [6]:
# Cell 6 — Generate manifest (web/images.json) and copy to web project

WEB_PROJECT_PATH = Path("/Users/jwatts/StudioProjects/git/IslandSkiesWeb/src/data")

def make_alt_text(row):
    """Generate descriptive alt text for accessibility."""
    if row["category"] == "dso":
        catalog = format_catalog_id(row["objectId"])
        name = row["displayName"]
        return f"{catalog} - {name}" if name != catalog else catalog
    return row["displayName"]


def make_cloudinary_id(row):
    folder = CLOUDINARY_FOLDER_MAP[row["category"]]
    return f"{CLOUDINARY_ROOT}/{folder}/{row['imageId']}"


# Gallery sort order — images with a sortOrder appear first (ascending).
# Add entries here to pin images to the top of their category.
SORT_ORDER = {
    "ic443": 1,
    "ngc1499": 2,
    "ngc2264": 3,
    "ngc2239": 4,
    "ngc2359": 5,
    "ic410-tadpoles": 6,
    "m45": 7,
    "m42-2": 8,
    "m42": 9,
    "m31": 10,
    "ngc4565":11,
    "m20": 12,
    "ngc2237": 13,
    "m16": 14,
    "m16-pillars-of-creation": 15,    
}

# Images to exclude from the gallery. Add imageIds here to hide them.
EXCLUDE = {
    "ic410",
    "ic434-2",
    "ic434-3",
    "m1",
    "m81m82"
}

manifest = []
excluded_count = 0
for _, row in images_df.iterrows():
    if row["imageId"] in EXCLUDE:
        excluded_count += 1
        continue
    entry = {
        "id": row["imageId"],
        "title": row["displayName"],
        "category": row["category"],
        "objectId": row["objectId"],
        "region": row["region"] if pd.notna(row.get("region")) else None,
        "cloudinaryId": make_cloudinary_id(row),
        "altText": make_alt_text(row),
        "objectName": row["displayName"],
        "catalogId": format_catalog_id(row["objectId"]) if row["category"] == "dso" else None,
        "ra": row["ra"] if pd.notna(row.get("ra")) else None,
        "dec": row["dec"] if pd.notna(row.get("dec")) else None,
        "constellation": row["constellation"] if pd.notna(row.get("constellation")) else None,
        "objectType": row["type"] if pd.notna(row.get("type")) else None,
        "objectSubType": row["subType"] if pd.notna(row.get("subType")) else None,
        "filename": f"{row['imageId']}.jpg",
        "width": int(row["width"]),
        "height": int(row["height"]),
        "sortOrder": SORT_ORDER.get(row["imageId"]),
    }
    manifest.append(entry)

manifest_json = json.dumps(manifest, indent=2)

manifest_path = OUTPUT_DIR / "images.json"
manifest_path.write_text(manifest_json)
print(f"Wrote {len(manifest)} entries to {manifest_path.resolve()}")
if excluded_count:
    print(f"Excluded {excluded_count} images: {EXCLUDE}")
print(f"Sort order applied to {sum(1 for e in manifest if e['sortOrder'])} images")

# Copy to web project
web_dest = WEB_PROJECT_PATH / "images.json"
web_dest.write_text(manifest_json)
print(f"Copied to {web_dest.resolve()}")

Wrote 52 entries to /Users/jwatts/StudioProjects/git/AstroPlannerData/web/images.json
Excluded 4 images: {'ic434-3', 'm81m82', 'ic434-2', 'm1', 'ic410'}
Sort order applied to 14 images
Copied to /Users/jwatts/StudioProjects/git/IslandSkiesWeb/src/data/images.json


In [7]:
# Cell 7 — Cloudinary upload (skipped when UPLOAD_TO_CLOUDINARY is False)

if not UPLOAD_TO_CLOUDINARY:
    print("Skipping Cloudinary upload (UPLOAD_TO_CLOUDINARY = False)")
else:
    import os
    from dotenv import load_dotenv
    import cloudinary
    import cloudinary.uploader

    load_dotenv()

    cloudinary.config(
        cloud_name=os.environ["CLOUDINARY_CLOUD_NAME"],
        api_key=os.environ["CLOUDINARY_API_KEY"],
        api_secret=os.environ["CLOUDINARY_API_SECRET"],
    )

    for _, row in images_df.iterrows():
        path = OUTPUT_DIR / row["category"] / f"{row['imageId']}.jpg"
        public_id = make_cloudinary_id(row)
        result = cloudinary.uploader.upload(
            str(path),
            public_id=public_id,
            overwrite=True,
            invalidate=True,
            resource_type="image",
        )
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  {row['imageId']:<35} {size_mb:.1f} MB  →  {result['secure_url']}")

    print(f"\nUploaded {len(images_df)} images to Cloudinary (with CDN invalidation).")

  ic1805                              8.5 MB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1775892805/island-skies-astro/dso/ic1805.jpg
  ic1805-ic1805-inner                 1.6 MB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1775892811/island-skies-astro/dso/ic1805-ic1805-inner.jpg
  ic410                               1.6 MB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1775892816/island-skies-astro/dso/ic410.jpg
  ic410-tadpoles                      1.6 MB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1775892823/island-skies-astro/dso/ic410-tadpoles.jpg
  ic434-2                             0.9 MB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1776057348/island-skies-astro/dso/ic434-2.jpg
  ic434-3                             2.9 MB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1776057356/island-skies-astro/dso/ic434-3.jpg
  ic434                               1.7 MB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1

In [8]:
# Cell 8 — Constellation context charts for all DSO objects

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "astroquery"])

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.coordinates import SkyCoord
import astropy.units as u
from astroquery.vizier import Vizier
import requests

# --- Configuration ---
FOV_DEG = 25
QUERY_RADIUS = 45
MAG_LIMIT = 5
CHART_SIZE = (10, 10)
CHART_DPI = 150
CHART_OUTPUT = Path("web/charts")
CHART_OUTPUT.mkdir(parents=True, exist_ok=True)
BG = "#0d1117"


def gnomonic(ra, dec, ra0, dec0):
    ra_r, dec_r = np.radians(ra), np.radians(dec)
    ra0_r, dec0_r = np.radians(ra0), np.radians(dec0)
    cos_c = (np.sin(dec0_r) * np.sin(dec_r) +
             np.cos(dec0_r) * np.cos(dec_r) * np.cos(ra_r - ra0_r))
    x = np.cos(dec_r) * np.sin(ra_r - ra0_r) / cos_c
    y = (np.cos(dec0_r) * np.sin(dec_r) -
         np.sin(dec0_r) * np.cos(dec_r) * np.cos(ra_r - ra0_r)) / cos_c
    return np.degrees(x), np.degrees(y)


# --- Load DSO catalog and get unique DSO objects with RA/Dec ---
with open("mobile/dso.json") as f:
    dso_list = json.load(f)
dso_by_id = {obj["objectId"]: obj for obj in dso_list}

dso_object_ids = images_df[images_df["category"] == "dso"]["objectId"].unique()
targets = []
for oid in sorted(set(dso_object_ids)):
    if oid in dso_by_id and dso_by_id[oid].get("ra") and dso_by_id[oid].get("dec"):
        targets.append(dso_by_id[oid])
print(f"{len(targets)} DSO objects to chart")

# --- Fetch Stellarium constellation data (once) ---
CONST_URL = ("https://raw.githubusercontent.com/Stellarium/stellarium/"
             "master/skycultures/modern/index.json")
resp = requests.get(CONST_URL, timeout=15)
resp.raise_for_status()
sky_data = resp.json()

const_hip_ids = set()
for constellation in sky_data.get("constellations", []):
    for polyline in constellation.get("lines", []):
        const_hip_ids.update(polyline)

print("Stellarium data loaded. Generating charts...\n")

viz = Vizier(columns=["HIP", "_RAJ2000", "_DEJ2000", "Vmag"], row_limit=-1)

for target in targets:
    object_id = target["objectId"]
    display_name = target["displayName"]
    ra_deg = target["ra"] * 15
    dec_deg = target["dec"]
    target_coord = SkyCoord(ra=ra_deg * u.deg, dec=dec_deg * u.deg)

    # Fetch stars for this region
    result = viz.query_region(target_coord, radius=QUERY_RADIUS * u.deg, catalog="I/239/hip_main")
    hip_table = result[0]
    bright = hip_table[hip_table["Vmag"] < MAG_LIMIT]

    star_ra = np.array(bright["_RAJ2000"])
    star_dec = np.array(bright["_DEJ2000"])
    star_mag = np.array(bright["Vmag"])
    star_hip = np.array(bright["HIP"])
    hip_to_radec = {h: (r, d) for h, r, d in zip(star_hip, star_ra, star_dec)}

    sx, sy = gnomonic(star_ra, star_dec, ra_deg, dec_deg)

    # Build constellation lines for this region
    const_lines = []
    const_points = {}
    for constellation in sky_data.get("constellations", []):
        abbrev = constellation["id"].split()[-1]
        for polyline in constellation.get("lines", []):
            for i in range(len(polyline) - 1):
                h1, h2 = polyline[i], polyline[i + 1]
                if h1 not in hip_to_radec or h2 not in hip_to_radec:
                    continue
                r1, d1 = hip_to_radec[h1]
                r2, d2 = hip_to_radec[h2]
                x1, y1 = gnomonic(r1, d1, ra_deg, dec_deg)
                x2, y2 = gnomonic(r2, d2, ra_deg, dec_deg)
                if (max(abs(x1), abs(y1)) < FOV_DEG * 1.4 or
                    max(abs(x2), abs(y2)) < FOV_DEG * 1.4):
                    const_lines.append(((-x1, y1), (-x2, y2)))
                    const_points.setdefault(abbrev, []).extend([(-x1, y1), (-x2, y2)])

    # Plot
    fig, ax = plt.subplots(figsize=CHART_SIZE, facecolor=BG)
    ax.set_facecolor(BG)

    for (x1, y1), (x2, y2) in const_lines:
        ax.plot([x1, x2], [y1, y2], color="#2a5da8", linewidth=1.8, alpha=0.85, zorder=2)

    labeled = set()
    for abbrev, pts in const_points.items():
        if abbrev in labeled:
            continue
        xs, ys = zip(*pts)
        cx, cy = np.mean(xs), np.mean(ys)
        if abs(cx) < FOV_DEG * 0.85 and abs(cy) < FOV_DEG * 0.85:
            ax.text(cx, cy, abbrev, color="#8899bb", fontsize=11, alpha=0.7,
                    ha="center", va="center", style="italic", fontweight="bold", zorder=3)
            labeled.add(abbrev)

    is_const_star = np.array([h in const_hip_ids for h in star_hip])
    base_sizes = np.clip((MAG_LIMIT + 0.5 - star_mag), 0.3, None) ** 2.5 * 1.2

    ax.scatter(-sx[~is_const_star], sy[~is_const_star],
               s=base_sizes[~is_const_star], c="white", alpha=0.7, edgecolors="none", zorder=5)
    ax.scatter(-sx[is_const_star], sy[is_const_star],
               s=base_sizes[is_const_star] * 1.8, c="#a8c8ff", alpha=0.95, edgecolors="none", zorder=6)

    ax.plot(0, 0, "+", color="#ff5555", markersize=22, markeredgewidth=2.5, zorder=10)
    ax.annotate(display_name, (0, 0), xytext=(14, 14),
                textcoords="offset points", color="#ff5555", fontsize=12,
                fontweight="bold", zorder=10)

    ax.set_xlim(-FOV_DEG, FOV_DEG)
    ax.set_ylim(-FOV_DEG, FOV_DEG)
    ax.set_aspect("equal")
    ax.axis("off")

    plt.tight_layout()
    out_path = CHART_OUTPUT / f"{object_id}-context.png"
    fig.savefig(out_path, dpi=CHART_DPI, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)

    size_kb = out_path.stat().st_size / 1024
    print(f"  {object_id:<20} {display_name:<30} {len(const_lines):>3} lines  {size_kb:.0f} KB")

print(f"\nDone. {len(targets)} charts saved to {CHART_OUTPUT.resolve()}")

36 DSO objects to chart
Stellarium data loaded. Generating charts...

  barnard68            Ophiuchus Globule               78 lines  121 KB
  barnard72            Snake Nebula                    78 lines  123 KB
  ic1805               Heart Nebula                    39 lines  80 KB
  ic410                Tadpoles Nebula                 72 lines  112 KB
  ic434                Horsehead Nebula                96 lines  149 KB
  ic443                Jellyfish Nebula                78 lines  127 KB
  ic5146               Cocoon Nebula                   47 lines  85 KB
  m1                   Crab Nebula                     81 lines  131 KB
  m101                 Pinwheel Galaxy                 56 lines  89 KB
  m13                  Hercules Cluster                59 lines  82 KB
  m16                  Eagle Nebula                    65 lines  108 KB
  m20                  Trifid Nebula                   71 lines  117 KB
  m27                  Dumbbell Nebula                 46 lines  87 KB

In [9]:
# Cell 9 — Upload charts to Cloudinary & patch manifest with contextChart field

CHART_CLOUDINARY_FOLDER = "charts"

if not UPLOAD_TO_CLOUDINARY:
    print("Skipping chart upload (UPLOAD_TO_CLOUDINARY = False)")
    print("Patching manifest with contextChart references anyway...\n")
else:
    import os
    from dotenv import load_dotenv
    import cloudinary
    import cloudinary.uploader

    load_dotenv()

    cloudinary.config(
        cloud_name=os.environ["CLOUDINARY_CLOUD_NAME"],
        api_key=os.environ["CLOUDINARY_API_KEY"],
        api_secret=os.environ["CLOUDINARY_API_SECRET"],
    )

    chart_files = sorted(CHART_OUTPUT.glob("*-context.png"))
    print(f"Uploading {len(chart_files)} charts to Cloudinary...\n")

    for chart_path in chart_files:
        object_id = chart_path.stem.replace("-context", "")
        public_id = f"{CLOUDINARY_ROOT}/{CHART_CLOUDINARY_FOLDER}/{chart_path.stem}"
        result = cloudinary.uploader.upload(
            str(chart_path),
            public_id=public_id,
            overwrite=True,
            invalidate=True,
            resource_type="image",
        )
        size_kb = chart_path.stat().st_size / 1024
        print(f"  {chart_path.stem:<30} {size_kb:.0f} KB  →  {result['secure_url']}")

    print(f"\nUploaded {len(chart_files)} charts to Cloudinary.")

# --- Patch manifest with contextChart field ---
manifest_path = OUTPUT_DIR / "images.json"
manifest = json.loads(manifest_path.read_text())

chart_ids = {p.stem.replace("-context", "") for p in CHART_OUTPUT.glob("*-context.png")}

patched = 0
for entry in manifest:
    oid = entry["objectId"]
    if oid in chart_ids and entry["category"] == "dso":
        entry["contextChart"] = f"{CLOUDINARY_ROOT}/{CHART_CLOUDINARY_FOLDER}/{oid}-context"
        patched += 1
    else:
        entry.pop("contextChart", None)

manifest_json = json.dumps(manifest, indent=2)
manifest_path.write_text(manifest_json)
print(f"\nPatched {patched} manifest entries with contextChart")
print(f"Wrote {manifest_path.resolve()}")

web_dest = WEB_PROJECT_PATH / "images.json"
web_dest.write_text(manifest_json)
print(f"Copied to {web_dest.resolve()}")

Uploading 36 charts to Cloudinary...

  barnard68-context              121 KB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1776553972/island-skies-astro/charts/barnard68-context.png
  barnard72-context              123 KB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1776553973/island-skies-astro/charts/barnard72-context.png
  ic1805-context                 80 KB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1776553973/island-skies-astro/charts/ic1805-context.png
  ic410-context                  112 KB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1776553974/island-skies-astro/charts/ic410-context.png
  ic434-context                  149 KB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1776553975/island-skies-astro/charts/ic434-context.png
  ic443-context                  127 KB  →  https://res.cloudinary.com/jeffrwatts/image/upload/v1776553976/island-skies-astro/charts/ic443-context.png
  ic5146-context                 85 KB  →  https